In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import shap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')

sns.set_palette("RdBu_r")
sns.set_style("whitegrid", {'grid.linestyle': '--', 'axes.edgecolor': '0.3'})
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'figure.dpi': 600,
    'savefig.dpi': 600,
    'savefig.format': 'jpeg'
})

# Read dataset
data = pd.read_csv("01_All_data.csv")
X = data.drop(columns=["n"])
y = data["n"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# Train model
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)
xgb_model.fit(X_train, y_train)

In [ ]:
# Use the model to predict on training and test datasets
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

# Calculate evaluation metrics: RMSE, R²
# RMSE
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
rmse_train = rmse(y_train, y_train_pred)
rmse_test = rmse(y_test, y_test_pred)

# NSE (Nash‑Sutcliffe Efficiency)
def NSE(y_pred, y_true):
    ave_obs = sum(y_true)/len(y_true)
    Numerator = sum((y_true-y_pred)**2)
    Denominator = sum((y_true-ave_obs)**2)
    return 1 -Numerator/Denominator
nse_train = NSE(y_train, y_train_pred)
nse_test = NSE(y_test, y_test_pred)

# R²
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print("Training set evaluation:")
print(f"RMSE: {rmse_train:.4f}, R²: {r2_train:.4f}, NSE: {nse_train:.4f}")
print("Test set evaluation:")
print(f"RMSE: {rmse_test:.4f}, R²: {r2_test:.4f}, NSE: {nse_test:.4f}")

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test)

In [ ]:
# ====== Plot figures ======
# Feature importance overlayed with beeswarm plot
fig, ax1 = plt.subplots(figsize=(10, 8), dpi=600)

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=X_test.columns,
    plot_type="dot",
    #cmap='RdYlGn',
    show=False,
    color_bar=True
)

plt.gca().set_position([0.2, 0.2, 0.65, 0.65])
ax1 = plt.gca()

fig = plt.gcf()
cbar = fig.axes[-1]

# Adjust position and size ([left, bottom, width, height])
cbar.set_position([0.88, 0.2, 0.03, 0.65])  

cbar.tick_params(labelsize=14)

cbar.set_title('Feature value', fontsize=14, pad=12)

ax2 = ax1.twiny()

shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.gca().set_position([0.2, 0.2, 0.65, 0.65])  

ax2.axhline(y=14, color='gray', linestyle='-', linewidth=1)

bars = ax2.patches
for bar in bars:
    bar.set_alpha(0.2)

ax1.set_xlabel('Shapley Value Contribution (Bee Swarm)', fontsize=16)
ax2.set_xlabel('Mean Shapley Value (Feature Importance)', fontsize=16, labelpad=16)
ax2.xaxis.set_label_position('top')  
ax2.xaxis.tick_top()                
ax1.set_ylabel('Features', fontsize=16)

ax1.tick_params(axis='both', labelsize=14)
ax2.tick_params(axis='x', labelsize=14)

plt.tight_layout()
ax1.grid(False)
ax2.grid(False)

plt.show()

In [ ]:
# Settings for SHAP main‑effect and interaction‑effect plots with trend lines
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
import numpy as np

def add_polynomial_trendline_with_ci(x, y, ax, n_bootstrap=100, ci_color='lightblue'):
    sort_idx = np.argsort(x)
    x_sorted, y_sorted = x[sort_idx], y[sort_idx]
    x_trend = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
    bootstrap_predictions = []

    # Bootstrap sampling
    for _ in range(n_bootstrap):
        idx = resample(range(len(x_sorted)), replace=True)
        x_res, y_res = x_sorted[idx], y_sorted[idx]
        model = Pipeline([('poly', PolynomialFeatures(degree=3)),
                          ('linear', LinearRegression())])
        model.fit(x_res.reshape(-1, 1), y_res)
        bootstrap_predictions.append(model.predict(x_trend))
    
    bootstrap_predictions = np.array(bootstrap_predictions)
    lower = np.percentile(bootstrap_predictions, 2.5, axis=0)
    upper = np.percentile(bootstrap_predictions, 97.5, axis=0)

    # Final trend line
    model_final = Pipeline([('poly', PolynomialFeatures(degree=3)),
                            ('linear', LinearRegression())])
    model_final.fit(x_sorted.reshape(-1, 1), y_sorted)
    y_trend = model_final.predict(x_trend)
    
    ax.fill_between(x_trend.ravel(), lower, upper, color=ci_color, alpha=0.4, label='95% CI')
    ax.plot(x_trend, y_trend, color='darkblue', linewidth=2, label='Trend Line')
    ax.legend(loc='best', fontsize=20)

In [ ]:
# Plot SHAP main‑effect or interaction‑effect figure
from matplotlib.colors import LinearSegmentedColormap
shap_cmap = LinearSegmentedColormap.from_list(
    'shap_default',
    [
        (0.0, '#008BFB'), 
        (0.5, '#A61AA9'),
        (1.0, '#FF0052')  
    ]
)

def plot_feature_interaction(shap_values, features, feature_names, feature1_index, feature2_index, ci_color='lightblue'):
    fig, ax = plt.subplots(figsize=(15, 8))


    
    feature1_name = feature_names[feature1_index]
    feature2_name = feature_names[feature2_index]
    
    x = features[:, feature1_index]
    y = shap_values.values[:, feature1_index]
    color_values = features[:, feature2_index]
    
    # Feature display range (show 5%‑95% of data below)
    vmin = np.percentile(color_values, 5)
    vmax = np.percentile(color_values, 95)
    scatter = ax.scatter(x, y, c=color_values, cmap=shap_cmap,vmin=vmin, vmax=vmax, alpha=0.8, s=15)
    add_polynomial_trendline_with_ci(x, y, ax, ci_color=ci_color)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label(f'{feature2_name}', fontsize=20)
    
    cbar.ax.tick_params(labelsize=20)
    
    ax.set_xlabel(feature1_name, fontsize=22)
    ax.set_ylabel('SHAP Value', fontsize=22)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    #ax.grid(False)

    plt.tight_layout()
    plt.show()

# Plot SHAP main‑effect or interaction‑effect figure. Two integers represent the (n‑1)‑th variable (0 stands for the first variable)
# If the two integers are equal, it represents the main‑effect
plot_feature_interaction(shap_values, X_test.values, X_test.columns.tolist(), 4, 3, ci_color='lightblue')